# Deep Agents

In [27]:
# Imports and environment first

import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_community.tools import GoogleSerperRun
from langchain_community.utilities import GoogleSerperAPIWrapper
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent

load_dotenv(override=True)

True

In [28]:
# Create Tool

client = MultiServerMCPClient({
    "playwright": {
        "transport": "stdio",
        "command": "npx",
        "args": ["-y", "@playwright/mcp@latest", "--isolated"],
    }
})

browser_tools = await client.get_tools()
print(f"Loaded {len(browser_tools)} browser tools:")
for t in browser_tools:
    print(" -", t.name)

Loaded 25 browser tools:
 - browser_close
 - browser_resize
 - browser_console_messages
 - browser_handle_dialog
 - browser_emulate_media
 - browser_evaluate
 - browser_file_upload
 - browser_drop
 - browser_find
 - browser_fill_form
 - browser_press_key
 - browser_type
 - browser_navigate
 - browser_navigate_back
 - browser_network_requests
 - browser_network_request
 - browser_run_code_unsafe
 - browser_take_screenshot
 - browser_snapshot
 - browser_click
 - browser_drag
 - browser_hover
 - browser_select_option
 - browser_tabs
 - browser_wait_for


In [29]:
browser_model = ChatOpenAI(
    model="openai/gpt-5.4-mini",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

browser_agent = {
    "name": "browser_agent",
    "description": (
        "Which F1 Driver was the most improved F1 Driver in the 2024 season?"
    ),
    "model": browser_model,
    "tools": browser_tools,
    "system_prompt": """
     "You are a research analyst. Plan your work with your todo tool, "
    "research with the browser tool, and write your findings as a tidy markdown briefing to a file."
    """
}

In [30]:
overall_instructions = """
You are an F1 Editor & your task is to read the findings from the browser sub-agent and write a report to be published on
the most improved F1 driver in 2024.
"""

In [31]:
model = ChatOpenAI(
    model="openai/gpt-5.4-mini",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)
lead = create_deep_agent(
    model=model,
    system_prompt=overall_instructions,
    subagents=[browser_agent],
    backend=FilesystemBackend(root_dir=sandbox, virtual_mode=True),
)

In [33]:
mission = """
Write a report to be published on the most improved F1 driver in 2024, this must be 
in a markdown file and saved as F1Report.md.
"""

result = await lead.ainvoke({"messages": [{"role": "user", "content": mission}]})

In [34]:
print(result["messages"][-1].content)

Done — I wrote the report to **`/F1Report.md`**.
